# AE5510 – Printable adversarial patch training

Curated from the supplied `uas-ab.ipynb`. Includes session setup, printable palette/NPS machinery, EOT transformations, patch optimisation, size-matched controls, robustness sweeps, figures and clamp correction. Run in order within this experiment group in Kaggle with GPU and required inputs/checkpoints.

## Printable patch workflow – source cell 41

In [ ]:
# ===== CELL S: SETUP (run FIRST in every session; ~3 min, no training) =====
import os, re, json, time, random, shutil, subprocess, sys, collections, math, copy
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
import numpy as np, cv2, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from ultralytics import YOLO
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then re-run"
dev = torch.device("cuda")
NAMES = {0: "Aircraft", 1: "Building", 2: "Military Truck", 3: "Tank"}
ALL_TYPES = ["flip", "brightness", "haze", "shadow", "motionblur", "zoom_in", "zoom_out", "rotation"]
AUG = dict(mixup=0.15, scale=0.7, degrees=10, translate=0.2, hsv_s=0.8, hsv_v=0.5)

# ---- find inputs ----
DATA_ROOT, found = None, []
for root, dirs, files in os.walk("/kaggle/input"):
    if "Training Data" in dirs and DATA_ROOT is None:
        DATA_ROOT = root
    if "det96_final.pt" in files:
        found.append(os.path.join(root, "det96_final.pt"))
assert DATA_ROOT, "Add Input -> uas-aerial"
DET96 = sorted(found, key=lambda q: "FINAL_PROJECT_BACKUP" not in q)[0] if found else None
print("data    :", DATA_ROOT)
print("detector:", DET96 or "NOT FOUND (will retrain, about 9 min)")

# ---- copy data, clean labels ----
OUT = "/kaggle/working/yolo_data"
if os.path.exists(OUT):
    shutil.rmtree(OUT)
for key, folder in {"train": "Training Data", "val": "Validation Data", "test": "Testing Data"}.items():
    os.makedirs(f"{OUT}/images/{key}"); os.makedirs(f"{OUT}/labels/{key}")
    for f in os.listdir(f"{DATA_ROOT}/{folder}/Images"):
        shutil.copy(f"{DATA_ROOT}/{folder}/Images/{f}", f"{OUT}/images/{key}/{f}")
    for f in os.listdir(f"{DATA_ROOT}/{folder}/Labels"):
        shutil.copy(f"{DATA_ROOT}/{folder}/Labels/{f}", f"{OUT}/labels/{key}/{f}")

def clean_label_file(path):
    rows, changed = [], False
    for line in open(path):
        p = line.split()
        if not p:
            continue
        try:
            c = int(float(p[0])); v = [float(x) for x in p[1:]]
        except ValueError:
            changed = True; continue
        if len(v) == 4:
            x1, y1, x2, y2 = v[0]-v[2]/2, v[1]-v[3]/2, v[0]+v[2]/2, v[1]+v[3]/2
        elif len(v) >= 6 and len(v) % 2 == 0:
            xs, ys = v[0::2], v[1::2]
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys); changed = True
        else:
            changed = True; continue
        cx1, cy1, cx2, cy2 = max(0, x1), max(0, y1), min(1, x2), min(1, y2)
        if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
            changed = True
        w, h = cx2 - cx1, cy2 - cy1
        if w < 0.002 or h < 0.002:
            changed = True; continue
        rows.append(f"{c} {(cx1+cx2)/2:.6f} {(cy1+cy2)/2:.6f} {w:.6f} {h:.6f}")
    return rows, changed

for split in ["train", "val", "test"]:
    d = f"{OUT}/labels/{split}"
    for f in os.listdir(d):
        rows, changed = clean_label_file(f"{d}/{f}")
        if changed:
            open(f"{d}/{f}", "w").write("\n".join(rows) + ("\n" if rows else ""))

# ---- same 96 / 24 photo split as before (fixed seed 7) ----
def source_key(fname):
    m = re.match(r'^(.*?\.rf\.[0-9a-f]+)', fname)
    return m.group(1) if m else os.path.splitext(fname)[0]
def aug_type(fname):
    rest = os.path.splitext(fname)[0][len(source_key(fname)):]
    return re.sub(r'_\d+$', '', rest).lstrip('_') or "original"

scenes, photos = [], collections.defaultdict(list)
for f in sorted(os.listdir(f"{OUT}/images/train")):
    photos[source_key(f)].append(f)
for k, files in photos.items():
    taken, tr = collections.Counter(), []
    for f in files:
        t = aug_type(f)
        if t in ALL_TYPES and taken[t] < 1:
            tr.append(("train", f)); taken[t] += 1
    flips = [f for f in files if aug_type(f) == "flip"]
    scenes.append(dict(cls=k.split("-")[0].lower(), files=tr, eval=("train", (flips or files)[0])))
for split in ["val", "test"]:
    for f in sorted(os.listdir(f"{OUT}/images/{split}")):
        scenes.append(dict(cls=f.split("-")[0].lower(), files=[(split, f)], eval=(split, f)))
rng, by = random.Random(7), collections.defaultdict(list)
for s in scenes:
    by[s["cls"]].append(s)
train_sc, test_sc = [], []
for c, lst in by.items():
    lst = lst[:]; rng.shuffle(lst)
    test_sc += lst[:6]; train_sc += lst[6:]

def load_item(split, fname):
    img = cv2.cvtColor(cv2.imread(f"{OUT}/images/{split}/{fname}"), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    r = min(640 / H, 640 / W)
    nw, nh = int(round(W * r)), int(round(H * r))
    if (nw, nh) != (W, H):
        img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR if r > 1 else cv2.INTER_AREA)
    dw, dh = (640 - nw) / 2, (640 - nh) / 2
    left, top = int(round(dw - 0.1)), int(round(dh - 0.1))
    canvas = np.full((640, 640, 3), 114, np.uint8)
    canvas[top:top + nh, left:left + nw] = img
    objs = []
    for line in open(f"{OUT}/labels/{split}/{os.path.splitext(fname)[0]}.txt"):
        p = line.split()
        if len(p) != 5:
            continue
        c = int(float(p[0])); xc, yc, w, h = [float(v) for v in p[1:]]
        objs.append((c, (xc-w/2)*W*r+left, (yc-h/2)*H*r+top, (xc+w/2)*W*r+left, (yc+h/2)*H*r+top))
    return torch.from_numpy(canvas).permute(2, 0, 1).contiguous(), objs

train_pool = [load_item(sp, f) for sc in train_sc for (sp, f) in sc["files"]]
train_pool = [it for it in train_pool if it[1]]
test_pool = [load_item(*sc["eval"]) for sc in test_sc]
clean_imgs = [it[0].to(dev).float() / 255 for it in test_pool]
objs_list = [it[1] for it in test_pool]
print(f"patch-train: {len(train_pool)} images | patch-test: {len(test_pool)} images | test objects: {sum(len(o) for o in objs_list)}")

# ---- detector (honest YOLOv8s trained only on the 96 photos) ----
def retrain_detector():
    D96 = "/kaggle/working/det96_data"
    for s in ["train", "val"]:
        os.makedirs(f"{D96}/images/{s}", exist_ok=True); os.makedirs(f"{D96}/labels/{s}", exist_ok=True)
    def put(split, fname, dst, copies=1):
        stem, ext = os.path.splitext(fname)
        for c in range(copies):
            nm = stem if copies == 1 else f"{stem}_c{c}"
            shutil.copy(f"{OUT}/images/{split}/{fname}", f"{D96}/images/{dst}/{nm}{ext}")
            shutil.copy(f"{OUT}/labels/{split}/{stem}.txt", f"{D96}/labels/{dst}/{nm}.txt")
    for sc in train_sc:
        for (sp, f) in sc["files"]:
            put(sp, f, "train", 8 if len(sc["files"]) == 1 else 1)
    for sc in test_sc:
        put(sc["eval"][0], sc["eval"][1], "val")
    y = f"{D96}/data.yaml"
    open(y, "w").write(f"path: {D96}\ntrain: images/train\nval: images/val\n\nnames:\n  0: Aircraft\n  1: Building\n  2: Military Truck\n  3: Tank\n")
    YOLO("yolov8s.pt").train(data=y, epochs=50, patience=0, imgsz=640, batch=16, device=0, workers=2,
        project="/kaggle/working/honest", name="det96", exist_ok=True, seed=0, flipud=0.5, cos_lr=True,
        plots=False, val=False, **AUG)
    return YOLO("/kaggle/working/honest/det96/weights/last.pt")

det = YOLO(DET96) if DET96 else retrain_detector()
yolo = copy.deepcopy(det.model).float().to(dev).eval()      # private copy used for gradients
for p_ in yolo.parameters():
    p_.requires_grad_(False)

# ---- evaluation helpers ----
def iou_np(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0
def get_dets(res):
    b = res.boxes
    return list(zip(b.cls.int().tolist(), b.conf.tolist(), b.xyxy.tolist()))
def outcome(dets, o):
    c, x1, y1, x2, y2 = o
    best_same = best_other = 0.0
    for pc_, cf, pb in dets:
        i = iou_np((x1, y1, x2, y2), pb)
        if pc_ == c: best_same = max(best_same, i)
        else: best_other = max(best_other, i)
    if best_same >= 0.5: return "detected"
    if best_other >= 0.5: return "misclassified"
    if best_same >= 0.1: return "mislocalized"
    return "disappeared"
def predict_batch(imgs):
    return det.predict(source=torch.stack(imgs), conf=0.25, iou=0.7, imgsz=640, device=0, verbose=False)
def asr_eval(clean_list, adv_list, rc=None):
    rc = predict_batch(clean_list) if rc is None else rc
    ra = predict_batch(adv_list)
    cls_n = {c: 0 for c in NAMES}; cls_fail = {c: 0 for c in NAMES}
    kinds, per_image = collections.Counter(), []
    for objs, a_, b_ in zip(objs_list, rc, ra):
        da, db = get_dets(a_), get_dets(b_)
        n_i = f_i = 0
        for o in objs:
            if outcome(da, o) != "detected":
                continue
            n_i += 1; cls_n[o[0]] += 1
            k = outcome(db, o)
            if k != "detected":
                f_i += 1; cls_fail[o[0]] += 1; kinds[k] += 1
        per_image.append((n_i, f_i))
    n, fail = sum(cls_n.values()), sum(cls_fail.values())
    return dict(overall=(fail / n) if n else float("nan"), n=n,
                per_class={NAMES[c]: (cls_fail[c] / cls_n[c]) if cls_n[c] else float("nan") for c in NAMES},
                n_class={NAMES[c]: cls_n[c] for c in NAMES}, kinds=dict(kinds), per_image=per_image)
def write_eval_set(name, imgs):
    D = f"/kaggle/working/atk_eval/{name}"
    shutil.rmtree(D, ignore_errors=True)
    os.makedirs(f"{D}/images/val"); os.makedirs(f"{D}/labels/val")
    for i, (img, objs) in enumerate(zip(imgs, objs_list)):
        arr = (img.clamp(0, 1) * 255).round().byte().permute(1, 2, 0).cpu().numpy()
        cv2.imwrite(f"{D}/images/val/{i:03d}.png", cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))
        with open(f"{D}/labels/val/{i:03d}.txt", "w") as fh:
            for (c, x1, y1, x2, y2) in objs:
                fh.write(f"{c} {(x1+x2)/2/640:.6f} {(y1+y2)/2/640:.6f} {(x2-x1)/640:.6f} {(y2-y1)/640:.6f}\n")
    open(f"{D}/data.yaml", "w").write(f"path: {D}\ntrain: images/val\nval: images/val\n\nnames:\n  0: Aircraft\n  1: Building\n  2: Military Truck\n  3: Tank\n")
    return f"{D}/data.yaml"
def map_metrics(name, imgs):
    r = det.val(data=write_eval_set(name, imgs), split="val", imgsz=640, device=0, plots=False, verbose=False)
    return dict(P=float(r.box.mp), R=float(r.box.mr), map50=float(r.box.map50), map5095=float(r.box.map),
                ap50={NAMES[int(c)]: float(r.box.ap50[i]) for i, c in enumerate(r.box.ap_class_index)})

rc = predict_batch(clean_imgs)
clean_m = map_metrics("clean", clean_imgs)
a0 = asr_eval(clean_imgs, clean_imgs, rc=rc)
print(f"\nclean mAP50 on the 24 held-out photos: {clean_m['map50']:.3f}   (expect about 0.38)")
print(f"objects detected on the clean images : {a0['n']}   (expect about 130)")
print("READY")

## Printable patch workflow – source cell 42

In [ ]:
# ===== CELL P1: PRINTABLE, SIZE-LIMITED PATCH MACHINERY + CHECKS =====
PR = 64                                                     # patch resolution in pixels
LV = torch.linspace(0.08, 0.92, 6, device=dev)
PALETTE = torch.cartesian_prod(LV, LV, LV)                  # 216 printable colours (in printed space)
def gamut(x): return 0.08 + 0.84 * x                        # printers cannot make pure black / white

def nps_loss(p):                                            # non-printability score: distance to nearest printable colour
    px = gamut(p).permute(0, 2, 3, 1).reshape(-1, 3)
    return torch.cdist(px, PALETTE).min(1).values.mean()
def quantize(p):                                            # round every pixel to the printable palette
    with torch.no_grad():
        px = gamut(p).permute(0, 2, 3, 1).reshape(-1, 3)
        q = ((PALETTE[torch.cdist(px, PALETTE).argmin(1)] - 0.08) / 0.84).clamp(0, 1)
        return q.reshape(1, p.shape[2], p.shape[3], 3).permute(0, 3, 1, 2).contiguous()
def gblur(x, sigma):                                        # Gaussian blur, x: [1,C,H,W]
    k = int(2 * math.ceil(2 * sigma) + 1)
    ax = torch.arange(k, device=x.device).float() - k // 2
    g = torch.exp(-(ax ** 2) / (2 * sigma ** 2)); g = g / g.sum()
    C = x.shape[1]
    x = F.conv2d(F.pad(x, (k // 2, k // 2, 0, 0), mode="replicate"), g.view(1, 1, 1, k).repeat(C, 1, 1, 1), groups=C)
    return F.conv2d(F.pad(x, (0, 0, k // 2, k // 2), mode="replicate"), g.view(1, 1, k, 1).repeat(C, 1, 1, 1), groups=C)

def rand_params(n):
    U = lambda lo, hi: torch.empty(n, device=dev).uniform_(lo, hi)
    return dict(angle=U(-25, 25), scale=U(0.7, 1.1), shear=U(-0.15, 0.15), bright=U(-0.15, 0.15), contrast=U(0.8, 1.2))
def fixed_params(n, angle=0., scale=1., shear=0., bright=0., contrast=1.):
    f = lambda v: torch.full((n,), float(v), device=dev)
    return dict(angle=f(angle), scale=f(scale), shear=f(shear), bright=f(bright), contrast=f(contrast))
def warp_patch(patch, angle, scale, shear, bright, contrast):
    n = angle.shape[0]
    rgba = torch.cat([patch, torch.ones_like(patch[:, :1])], 1).expand(n, -1, -1, -1).contiguous()
    a_ = angle * math.pi / 180; cos, sin = torch.cos(a_), torch.sin(a_)
    theta = torch.zeros(n, 2, 3, device=patch.device)
    theta[:, 0, 0] = cos / scale; theta[:, 0, 1] = (-sin + shear) / scale
    theta[:, 1, 0] = sin / scale; theta[:, 1, 1] = cos / scale
    grid = F.affine_grid(theta, rgba.shape, align_corners=False)
    out = F.grid_sample(rgba, grid, align_corners=False, padding_mode="zeros")
    rgb = (out[:, :3] * contrast.view(n, 1, 1, 1) + bright.view(n, 1, 1, 1)).clamp(0, 1)
    return torch.cat([rgb, out[:, 3:4]], 1)

PC_TRAIN = dict(gamut=True, jitter=True, blur=True)         # print + camera simulation used in training
PC_EVAL = dict(gamut=True)                                  # nominal printed patch

def paste_patches(img, objs, patch, a, params=None, pc=None, rnd=False):
    """a = patch AREA as a fraction of each object's box: side = sqrt(a * w * h)."""
    if not objs:
        return img
    rgba = warp_patch(patch, **(params or rand_params(len(objs))))
    out = img.clone()
    for i, (c, x1, y1, x2, y2) in enumerate(objs):
        s = int(min(max(round(math.sqrt(a * (x2 - x1) * (y2 - y1))), 4), 128))
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        px = min(max(cx - s // 2, 0), 640 - s); py = min(max(cy - s // 2, 0), 640 - s)
        tile = F.interpolate(rgba[i:i + 1], size=(s, s), mode="bilinear", align_corners=False, antialias=True)[0]
        if pc is not None:
            rgb = tile[:3]
            if pc.get("gamut"):
                rgb = gamut(rgb)
            if pc.get("gain") is not None:                  # fixed colour shift (printer / camera)
                rgb = rgb * torch.tensor(pc["gain"], device=dev).view(3, 1, 1) + torch.tensor(pc.get("bias", (0, 0, 0)), device=dev).view(3, 1, 1)
            if rnd and pc.get("jitter"):                    # random colour shift (training)
                rgb = rgb * torch.empty(3, 1, 1, device=dev).uniform_(0.9, 1.1) + torch.empty(3, 1, 1, device=dev).uniform_(-0.05, 0.05)
            tile = torch.cat([rgb.clamp(0, 1), tile[3:4]], 0)
            sig = random.uniform(0, 1.0) if (rnd and pc.get("blur")) else pc.get("blur_sigma", 0)
            if sig and sig > 0.15:
                tile = gblur(tile[None], sig)[0]
        region = out[:, py:py + s, px:px + s].clone()
        out[:, py:py + s, px:px + s] = region * (1 - tile[3:4]) + tile[:3] * tile[3:4]
    return out

@torch.no_grad()
def apply_eval(img, objs, patch, a, pc=None, **fixed):
    return paste_patches(img, objs, patch, a, params=fixed_params(len(objs), **fixed), pc=pc)
def make_adv(patch, a, pc=PC_EVAL, imgs=None, **fixed):
    imgs = clean_imgs if imgs is None else imgs
    return [apply_eval(im, o, patch, a, pc, **fixed) for im, o in zip(imgs, objs_list)]

def xywh2xyxy(b):
    return torch.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2, b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], 1)
def box_iou_1(g, boxes):
    x1 = torch.maximum(g[0], boxes[:, 0]); y1 = torch.maximum(g[1], boxes[:, 1])
    x2 = torch.minimum(g[2], boxes[:, 2]); y2 = torch.minimum(g[3], boxes[:, 3])
    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    ag = (g[2] - g[0]) * (g[3] - g[1]); ab = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (ag + ab - inter + 1e-9)
def get_pred(x):
    out = yolo(x)
    pred = out[0] if isinstance(out, (tuple, list)) else out
    assert pred.shape[1] == 8, f"unexpected YOLO output shape {tuple(pred.shape)}"
    return pred
def suppression_loss(pred, objs, b, k=3, iou_t=0.3):
    boxes = xywh2xyxy(pred[b, :4, :].T); losses = []
    for (c, x1, y1, x2, y2) in objs:
        g = torch.tensor([x1, y1, x2, y2], device=pred.device, dtype=pred.dtype)
        near = (box_iou_1(g, boxes) > iou_t).to(pred.dtype)
        losses.append(torch.topk(pred[b, 4 + c, :] * near, k).values.mean())
    return torch.stack(losses)
def tv_loss(p):
    return (p[:, :, 1:, :] - p[:, :, :-1, :]).abs().mean() + (p[:, :, :, 1:] - p[:, :, :, :-1]).abs().mean()

def train_patch(a, mode="print", iters=1200, B=4, maxobj=10, lr=0.03, tv=0.1, nps_w=1.0, seed=0, log_every=300):
    torch.manual_seed(seed); random.seed(seed)
    patch = (torch.rand(1, 3, PR, PR, device=dev) * 0.5 + 0.25).requires_grad_(True)
    opt = torch.optim.Adam([patch], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=iters, eta_min=lr * 0.05)
    pc = PC_TRAIN if mode == "print" else None
    hist, t0 = [], time.time()
    for it in range(1, iters + 1):
        imgs, objl = [], []
        for (timg, objs) in random.sample(train_pool, B):
            o = objs if len(objs) <= maxobj else random.sample(objs, maxobj)
            im = paste_patches(timg.to(dev).float() / 255, o, patch, a, pc=pc, rnd=True)
            im = (im * random.uniform(0.7, 1.3)).clamp(0, 1)                       # scene lighting
            if mode == "print":
                im = (im + 0.03 * random.random() * torch.randn_like(im)).clamp(0, 1)   # sensor noise
            imgs.append(im); objl.append(o)
        pred = get_pred(torch.stack(imgs))
        det_loss = torch.cat([suppression_loss(pred, objl[b], b) for b in range(B)]).mean()
        loss = det_loss + tv * tv_loss(patch) + (nps_w * nps_loss(patch) if mode == "print" else 0.0)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        with torch.no_grad():
            patch.clamp_(0, 1)
        hist.append(det_loss.item())
        if it % log_every == 0:
            print(f"   iter {it}/{iters} | detection loss (last 50) {np.mean(hist[-50:]):.3f} | {(time.time()-t0)/60:.1f} min")
    return patch.detach(), hist

# ---------------- checks ----------------
tp = torch.rand(1, 3, PR, PR, device=dev, requires_grad=True)
timg, objs = train_pool[0]
im = paste_patches(timg.to(dev).float() / 255, objs[:10], tp, 0.10, pc=PC_TRAIN, rnd=True)
loss = suppression_loss(get_pred(im[None]), objs[:10], 0).mean() + 0.1 * tv_loss(tp) + nps_loss(tp)
loss.backward()
gm = tp.grad.abs().mean().item() if tp.grad is not None else 0.0
print("patch gradient: exists =", tp.grad is not None, "| mean |grad| =", gm)
assert gm > 0, "no gradient reaches the patch: send me this printout, do not train"
print(f"NPS of a random patch {nps_loss(tp).item():.3f} -> after rounding to the palette {nps_loss(quantize(tp.detach())).item():.4f}")
print("\npatch side at area = 10% of the object box (test objects), median over objects:")
for c in NAMES:
    v = [(math.sqrt(0.10 * (x2 - x1) * (y2 - y1)), math.sqrt(0.10 * (x2 - x1) * (y2 - y1)) / min(x2 - x1, y2 - y1))
         for objs in objs_list for (cc, x1, y1, x2, y2) in objs if cc == c]
    print(f"   {NAMES[c]:15s} {np.median([t[0] for t in v]):5.1f} px  = {np.median([t[1] for t in v]):.2f} x the object's shorter side  (n={len(v)})")
print("(the old patches were 0.50 x the shorter side, about 25% of the box area)")

## Printable patch workflow – source cell 43

In [ ]:
# ===== CELL P2: TRAIN SIZE-LIMITED PATCHES (printable, plus an unconstrained comparison) =====
ITERS = 1200
OUT2 = "/kaggle/working/final2"; PD = f"{OUT2}/patches"; os.makedirs(PD, exist_ok=True)
JOBS = [("print_a010_s0", 0.10, "print", 0), ("print_a010_s1", 0.10, "print", 1), ("print_a010_s2", 0.10, "print", 2),
        ("print_a005_s0", 0.05, "print", 0), ("print_a020_s0", 0.20, "print", 0),
        ("digital_a010_s0", 0.10, "digital", 0)]
patches, meta = {}, {}
for name, a, mode, sd in JOBS:
    f = f"{PD}/{name}.pt"
    if os.path.exists(f):
        patches[name] = torch.load(f, map_location="cpu")["patch"].to(dev); print("loaded", name)
    else:
        print(f"\n--- training {name} (area {a:.0%} of the object box, {mode}) ---")
        p, h = train_patch(a, mode, iters=ITERS, seed=sd)
        patches[name] = p
        torch.save({"patch": p.cpu(), "a": a, "mode": mode, "seed": sd, "loss": h}, f)
    meta[name] = dict(a=a, mode=mode, seed=sd)
for name in list(patches):                                  # the version that would actually be printed
    if meta[name]["mode"] == "print":
        patches[name + "_q"] = quantize(patches[name]); meta[name + "_q"] = dict(meta[name], q=True)
for n, p in patches.items():
    plt.imsave(f"{PD}/{n}.png", p[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy())
print("\nall patches trained / loaded:", list(patches))

## Printable patch workflow – source cell 44

In [ ]:
# ===== CELL P3: SIZE-MATCHED COMPARISON (trained vs gray / random of the SAME size) =====
SIZES = [0.05, 0.10, 0.20]
gray = torch.full((1, 3, PR, PR), 0.5, device=dev)
rands = [torch.rand(1, 3, PR, PR, device=dev, generator=torch.Generator(device=dev).manual_seed(s)) for s in (123, 124, 125)]
def ev(name, patch, a, pc):
    imgs = make_adv(patch, a, pc)
    r = asr_eval(clean_imgs, imgs, rc=rc); r.pop("per_image")
    return dict(map_metrics(name, imgs), asr=r)
R, NAMED = {}, {}
def add(a, cond, res, name=None):
    R.setdefault((a, cond), []).append(res)
    if name: NAMED[name] = res
for a in SIZES:
    add(a, "gray square", ev(f"gray_{a}", gray, a, PC_EVAL))
    for i, r_ in enumerate(rands):
        add(a, "random noise", ev(f"rand{i}_{a}", r_, a, PC_EVAL))
for name, mt in meta.items():
    if mt["mode"] == "digital":
        add(mt["a"], "digital (unconstrained)", ev(name, patches[name], mt["a"], None), name)
    elif mt.get("q"):
        add(mt["a"], "print (rounded to palette)", ev(name, patches[name], mt["a"], PC_EVAL), name)
    else:
        add(mt["a"], "print (continuous)", ev(name, patches[name], mt["a"], PC_EVAL), name)

def stat(lst, f):
    v = np.array([f(r) for r in lst], float); return np.nanmean(v), np.nanstd(v)
CONDS = ["gray square", "random noise", "digital (unconstrained)", "print (continuous)", "print (rounded to palette)"]
print("=" * 105)
print(f"CLEAN (24 unseen photos): P {clean_m['P']:.3f} R {clean_m['R']:.3f} mAP50 {clean_m['map50']:.3f}   |  n = {a0['n']} objects found on clean images")
for a in SIZES:
    print(f"\n=== patch area = {a:.0%} of the object box ===")
    rnd_asr = stat(R[(a, "random noise")], lambda r: r["asr"]["overall"])[0]
    for c in CONDS:
        if (a, c) in R:
            am, asd = stat(R[(a, c)], lambda r: r["asr"]["overall"]); mm, msd = stat(R[(a, c)], lambda r: r["map50"])
            print(f"{c:28s} ASR {am:.2f} +-{asd:.2f} | mAP50 {mm:.3f} | lift over random {am-rnd_asr:+.2f} | seeds {len(R[(a, c)])}")

main = R[(0.10, "print (rounded to palette)")]
kinds = collections.Counter()
for r in main: kinds.update(r["asr"]["kinds"])
tot = sum(kinds.values()) or 1
print(f"\nMAIN RESULT (area 10%, printable, rounded, 3 seeds): failure types disappeared/misclassified/mislocalized = "
      f"{kinds['disappeared']/tot:.0%} / {kinds['misclassified']/tot:.0%} / {kinds['mislocalized']/tot:.0%}")
nc = main[0]["asr"]["n_class"]
print(f"\nPER-CLASS ASR at area 10% (n = objects found on clean images)")
print(f"{'':28s}" + "".join(f"{NAMES[c][:9] + ' n=' + str(nc[NAMES[c]]):>16s}" for c in NAMES))
for c_ in ["gray square", "random noise", "print (rounded to palette)"]:
    print(f"{c_:28s}" + "".join(f"{np.nanmean([r['asr']['per_class'][NAMES[c]] for r in R[(0.10, c_)]]):16.2f}" for c in NAMES))
print("\nper-seed ASR (print, rounded, 10%):", {n: round(NAMED[n]["asr"]["overall"], 2) for n in NAMED if n.startswith("print_a010") and n.endswith("_q")})
json.dump({"clean": clean_m, "R": {f"{k[0]}|{k[1]}": v for k, v in R.items()}}, open(f"{OUT2}/p3_results.json", "w"), indent=1, default=float)

## Printable patch workflow – source cell 45

In [ ]:
# ===== CELL P4: SWEEPS FOR THE MAIN PRINTABLE PATCH (area 10%) =====
A0 = 0.10
MAIN = [patches[f"print_a010_s{s}_q"] for s in range(3)]
def d_gain(im, v): return (im * v).clamp(0, 1)
def d_noise(im, s):
    g = torch.Generator(device=im.device); g.manual_seed(5)
    return (im + s * torch.randn(im.shape, device=im.device, generator=g)).clamp(0, 1)
def d_jpeg(im, q):
    arr = (im.clamp(0, 1) * 255).round().byte().permute(1, 2, 0).cpu().numpy()
    ok, enc = cv2.imencode(".jpg", cv2.cvtColor(arr, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(q)])
    return torch.from_numpy(cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float().to(im.device) / 255
def whole(distort):
    return lambda v: ([distort(im, v) for im in clean_imgs], lambda p: [distort(x, v) for x in make_adv(p, A0, PC_EVAL)])
SHIFT = {"none": PC_EVAL,
         "warm": dict(gamut=True, gain=(1.1, 1.0, 0.9), bias=(0.03, 0.0, -0.03)),
         "cool": dict(gamut=True, gain=(0.9, 1.0, 1.1), bias=(-0.03, 0.0, 0.03))}
SW = [
    ("patch area, trained at 10% (fraction of object box)", [0.05, 0.10, 0.15, 0.20, 0.25], lambda v: (clean_imgs, lambda p: make_adv(p, v, PC_EVAL))),
    ("rotation (degrees)", [-30, -15, 0, 15, 30], lambda v: (clean_imgs, lambda p: make_adv(p, A0, PC_EVAL, angle=v))),
    ("scale / distance (trained 0.7-1.1)", [0.5, 0.7, 0.85, 1.0, 1.1], lambda v: (clean_imgs, lambda p: make_adv(p, A0, PC_EVAL, scale=v))),
    ("scene lighting (gain)", [0.6, 0.8, 1.0, 1.2, 1.4], whole(d_gain)),
    ("PRINT: colour shift", ["none", "warm", "cool"], lambda v: (clean_imgs, lambda p: make_adv(p, A0, SHIFT[v]))),
    ("CAMERA: blur (sigma px)", [0, 1, 2], lambda v: (clean_imgs, lambda p: make_adv(p, A0, dict(gamut=True, blur_sigma=v)))),
    ("CAMERA: sensor noise (sigma)", [0.0, 0.02, 0.05], whole(d_noise)),
    ("CAMERA: JPEG quality", [90, 50, 20], whole(d_jpeg)),
]
SWEEPS = {}
for title, values, build in SW:
    print("\n" + title)
    print(f"{'value':>8s}{'print':>8s}{'random':>8s}{'lift':>8s}   n")
    rows = {"values": values, "print": [], "random": [], "n": []}
    for v in values:
        cl, adv_fn = build(v)
        rc_ = predict_batch(cl)
        pr = [asr_eval(cl, adv_fn(p), rc=rc_) for p in MAIN]
        rn = [asr_eval(cl, adv_fn(p), rc=rc_) for p in rands]
        a_p = float(np.nanmean([x["overall"] for x in pr])); a_r = float(np.nanmean([x["overall"] for x in rn]))
        rows["print"].append(a_p); rows["random"].append(a_r); rows["n"].append(pr[0]["n"])
        print(f"{str(v):>8s}{a_p:8.2f}{a_r:8.2f}{a_p-a_r:8.2f}   {pr[0]['n']}")
    SWEEPS[title] = rows
json.dump(SWEEPS, open(f"{OUT2}/p4_sweeps.json", "w"), indent=1)

## Printable patch workflow – source cell 46

In [ ]:
# ===== CELL P5: PRINT-READY PATCH + FIGURES + ZIP =====
import zipfile, base64
from IPython.display import display, HTML
FIG = f"{OUT2}/figs"; PRT = f"{OUT2}/print_ready"; os.makedirs(FIG, exist_ok=True); os.makedirs(PRT, exist_ok=True)

# ---- print-ready patch: best of the 3 seeds, rounded to the palette ----
best = max([f"print_a010_s{s}_q" for s in range(3)], key=lambda n: NAMED[n]["asr"]["overall"])
bp = patches[best][0].permute(1, 2, 0).cpu().numpy()
printed = gamut(torch.tensor(bp)).numpy()                                   # colours as they should appear on paper
big = cv2.resize((printed * 255).round().astype(np.uint8), (1024, 1024), interpolation=cv2.INTER_NEAREST)
cv2.imwrite(f"{PRT}/patch_print_1024.png", cv2.cvtColor(big, cv2.COLOR_RGB2BGR))
sw = (PALETTE.cpu().numpy() * 255).round().astype(np.uint8)[::6].reshape(6, 6, 3)
cv2.imwrite(f"{PRT}/palette_swatches.png", cv2.cvtColor(cv2.resize(sw, (360, 360), interpolation=cv2.INTER_NEAREST), cv2.COLOR_RGB2BGR))
with open(f"{PRT}/PRINT_SPEC.txt", "w") as fh:
    fh.write(f"Printable adversarial patch (best seed: {best})\n")
    fh.write(f"- pattern: {PR}x{PR} cells, every cell one of 216 palette colours (6 levels per RGB channel between 0.08 and 0.92)\n")
    fh.write("- print on matte paper, no gloss; keep the print flat and facing the camera\n")
    fh.write("- size rule: side = sqrt(0.10 x object box area), centred on the object (at most 10% of the box area)\n")
    for c in NAMES:
        v = [(math.sqrt(0.10 * (x2 - x1) * (y2 - y1)) / min(x2 - x1, y2 - y1)) for objs in objs_list for (cc, x1, y1, x2, y2) in objs if cc == c]
        fh.write(f"- median side on our test objects, {NAMES[c]}: {np.median(v):.2f} x the object's shorter side\n")
    fh.write("- simulated during training: rotation +-25 deg, scale 0.7-1.1, shear, lighting, colour shift +-10%, blur up to 1 px, sensor noise\n")
print(open(f"{PRT}/PRINT_SPEC.txt").read())

# ---- Fig 1: ASR vs patch area (trained at that size) vs controls of the same size ----
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for cond, col, st in (("print (rounded to palette)", "#c62828", "o-"), ("random noise", "#f2a900", "s--"), ("gray square", "#9aa0a6", "^:")):
    xs = [a for a in SIZES if (a, cond) in R]
    m = [stat(R[(a, cond)], lambda r: r["asr"]["overall"]) for a in xs]
    ax.errorbar([f"{a:.0%}" for a in xs], [t[0] for t in m], yerr=[t[1] for t in m], fmt=st, color=col, capsize=3, label=cond)
ax.set_xlabel("patch area (fraction of the object's box)"); ax.set_ylabel("attack success rate"); ax.set_ylim(0, 1.05); ax.legend()
ax.set_title("Attack success vs patch size (24 unseen photos)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig1_asr_vs_size.png", dpi=150); plt.show()

# ---- Fig 2: all conditions at 10% ----
fig, ax = plt.subplots(figsize=(8.5, 4.4))
cs = [c for c in CONDS if (0.10, c) in R]
am = [stat(R[(0.10, c)], lambda r: r["asr"]["overall"]) for c in cs]
ax.bar(range(len(cs)), [t[0] for t in am], yerr=[t[1] for t in am], capsize=3, color=["#9aa0a6", "#f2a900", "#1565c0", "#e57373", "#c62828"][:len(cs)])
ax.set_xticks(range(len(cs))); ax.set_xticklabels([c.replace(" (", "\n(") for c in cs], fontsize=8); ax.set_ylim(0, 1.05); ax.set_ylabel("attack success rate")
for i, t in enumerate(am): ax.text(i, t[0] + 0.03, f"{t[0]:.2f}", ha="center")
ax.set_title("Patch area 10% of the object box")
plt.tight_layout(); plt.savefig(f"{FIG}/fig2_conditions_10pct.png", dpi=150); plt.show()

# ---- Fig 3: before / after with the print-ready patch ----
adv = make_adv(patches[best], 0.10, PC_EVAL)
pi = asr_eval(clean_imgs, adv, rc=rc)["per_image"]
cand = sorted([(f / n, i) for i, (n, f) in enumerate(pi) if n >= 2], reverse=True)
if cand:
    picks = [cand[0][1], cand[-1][1]] if len(cand) >= 2 else [cand[0][1]]
    rr_c = predict_batch([clean_imgs[i] for i in picks]); rr_a = predict_batch([adv[i] for i in picks])
    fig, ax = plt.subplots(len(picks), 2, figsize=(14, 7 * len(picks)), squeeze=False)
    for k, i in enumerate(picks):
        ax[k, 0].imshow(cv2.cvtColor(rr_c[k].plot(), cv2.COLOR_BGR2RGB)); ax[k, 0].axis("off"); ax[k, 0].set_title(f"clean ({pi[i][0]} objects found; numbers = detection confidence)")
        ax[k, 1].imshow(cv2.cvtColor(rr_a[k].plot(), cv2.COLOR_BGR2RGB)); ax[k, 1].axis("off"); ax[k, 1].set_title(f"printable patch, 10% of box ({pi[i][1]} lost)")
    plt.tight_layout(); plt.savefig(f"{FIG}/fig3_before_after.png", dpi=110); plt.show()

# ---- Fig 4: sweeps ----
fig, axs = plt.subplots(3, 3, figsize=(15, 11))
for ax in axs.ravel(): ax.axis("off")
for ax, (title, r) in zip(axs.ravel(), SWEEPS.items()):
    ax.axis("on"); xs = list(range(len(r["values"])))
    ax.plot(xs, r["print"], "o-", color="#c62828", label="printable patch"); ax.plot(xs, r["random"], "s--", color="#f2a900", label="random")
    ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in r["values"]]); ax.set_title(title, fontsize=8); ax.set_ylim(0, 1.05); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{FIG}/fig4_sweeps.png", dpi=130); plt.show()

# ---- Fig 5: the patches ----
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (n, t) in zip(axs, [("digital_a010_s0", "unconstrained"), ("print_a010_s0", "printable (continuous)"), ("print_a010_s0_q", "printable (rounded)")]):
    ax.imshow(gamut(patches[n][0].permute(1, 2, 0)).clamp(0, 1).cpu().numpy() if "digital" not in n else patches[n][0].permute(1, 2, 0).cpu().numpy())
    ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG}/fig5_patches.png", dpi=150); plt.show()

# ---- zip + download ----
OUT_ZIP = "/kaggle/working/review3_results.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT2):
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT2))
b = base64.b64encode(open(OUT_ZIP, "rb").read()).decode()
display(HTML(f'<a download="review3_results.zip" href="data:application/zip;base64,{b}">Click here to download review3_results.zip</a>'))
print("zip size MB:", round(os.path.getsize(OUT_ZIP) / 1e6, 2))

## Printable patch workflow – source cell 47

In [ ]:
# ===== CELL P6: CLAMP FIX + RE-RUN THE TWO AFFECTED RESULTS =====
if "_pp" not in globals():
    _pp = paste_patches                                    # keep the original, wrap once
def paste_patches(img, objs, patch, a, params=None, pc=None, rnd=False):
    return _pp(img, objs, patch, a, params, pc, rnd).clamp(0, 1)
def predict_batch(imgs):
    return det.predict(source=torch.stack(imgs).clamp(0, 1), conf=0.25, iou=0.7, imgsz=640, device=0, verbose=False)

PD = "/kaggle/working/final2/patches"
if not os.path.exists(f"{PD}/print_a010_s0.pt"):
    cand = [r for r, d, fs in os.walk("/kaggle/input") if "print_a010_s0.pt" in fs]
    assert cand, "Add the review3_results zip as a Kaggle dataset input first"
    PD = cand[0]
patches = {n: torch.load(f"{PD}/{n}.pt", map_location="cpu")["patch"].to(dev)
           for n in ["print_a010_s0", "print_a010_s1", "print_a010_s2", "digital_a010_s0"]}
for s in range(3):
    patches[f"print_a010_s{s}_q"] = quantize(patches[f"print_a010_s{s}"])
gray = torch.full((1, 3, PR, PR), 0.5, device=dev)
rands = [torch.rand(1, 3, PR, PR, device=dev, generator=torch.Generator(device=dev).manual_seed(s)) for s in (123, 124, 125)]
SHIFT = {"none": PC_EVAL,
         "warm": dict(gamut=True, gain=(1.1, 1.0, 0.9), bias=(0.03, 0.0, -0.03)),
         "cool": dict(gamut=True, gain=(0.9, 1.0, 1.1), bias=(-0.03, 0.0, 0.03))}

# 1) unconstrained (digital) patch, area 10%
imgs = make_adv(patches["digital_a010_s0"], 0.10, None)
r = asr_eval(clean_imgs, imgs, rc=rc)
print("max pixel value in attacked images:", max(x.max().item() for x in imgs), "(must be <= 1.0)")
print(f"\nDIGITAL (unconstrained) patch, area 10%: ASR {r['overall']:.2f} (n={r['n']}) | failure types {r['kinds']}")
print("   per class:", {k: round(v, 2) for k, v in r["per_class"].items()})

# 2) colour shift: print patches vs random
print("\nPRINT colour shift (area 10%)")
print(f"{'shift':>8s}{'print':>8s}{'random':>8s}{'lift':>8s}")
for v in ["none", "warm", "cool"]:
    pr = [asr_eval(clean_imgs, make_adv(patches[f"print_a010_s{s}_q"], 0.10, SHIFT[v]), rc=rc)["overall"] for s in range(3)]
    rn = [asr_eval(clean_imgs, make_adv(p, 0.10, SHIFT[v]), rc=rc)["overall"] for p in rands]
    print(f"{v:>8s}{np.nanmean(pr):8.2f}{np.nanmean(rn):8.2f}{np.nanmean(pr)-np.nanmean(rn):8.2f}")